# Ek 5: Dogrulamada Tanimli Hata Skoru ve Uc Tohumlu Egitim

Bu not defteri ana sonuclari degistirmez, iki acik soruyu olcer.

1. **Hata skoru (yeniden egitim yok, ilk sonuc):** Mevcut guvenilirlik katsayisi yerine, modelin en yuksek olasiligi ve maske alanindan
   olusan skor **yalnizca dogrulama hastalarinda** tanimlanir ve testte mevcut katsayi ile max(P) ile karsilastirilir.
   Hata tanimi makaledeki gibidir: Dice < 0,5.
2. **Uc tohum (en uzun adim):** Tek gorevli ve cok gorevli modeller iki yeni tohumla (7 ve 123) ayni ayarlarla yeniden egitilir,
   tohum 42 mevcut agirliklarla ayni yolla degerlendirilir. Agirliklar Drive'a yazilir, baglanti koparsa **Tumunu calistir**
   tamamlanan modelleri atlar.

Calistirma: GPU secin ve **Tumunu calistir** deyin. Yalnizca skor icin `RUN_TOHUM = False` yapin. Ciktilar Drive'da
`pnx_ek_analizler` klasorune yazilir: `TABLO_ek5_skor.csv`, `TABLO_ek5_tohum.csv`, `TABLO_ek5_tohum_ozet.csv`.

In [ ]:
!pip install -q segmentation_models_pytorch==0.5.0 albumentations opencv-python-headless pillow scipy matplotlib pandas scikit-learn

### Ortak yardimci modul
Asagidaki hucre `pnx_ortak.py` dosyasini Colab diskine yazar. Degistirmeniz gerekmez.

In [ ]:
%%writefile /content/pnx_ortak.py
# -*- coding: utf-8 -*-
"""
pnx_ortak.py - Pnomotoraks BT analizi icin ortak yardimci modul.

Veri hazirlama, hasta bazli bolme, U-Net cikarimi (kayan pencere ve tam cozunurluk), SAM2 goruntu ve
video modu, birlestirme kurali, guvenilirlik katsayisi ve metrik fonksiyonlarini icerir.
Sabit tohumla (SEED=42) hasta bazli egitim/dogrulama/test bolmesi uretilir.

Colab'da kullanim:
    import sys; sys.path.insert(0, '/content')
    import pnx_ortak as po
    po.init_unet(); po.init_sam2_image()
"""
import os, glob, random, time
import numpy as np
import cv2
import torch
from PIL import Image
from scipy import ndimage

# ---------------------------------------------------------------------------
# Yapilandirma
# ---------------------------------------------------------------------------
RAW_DIR = '/content/drive/MyDrive/CT_goruntuler_anon'
DRIVE_BACKUP_DIR = '/content/drive/MyDrive/pnx_artifacts'
UNET_WEIGHT_PATH = f'{DRIVE_BACKUP_DIR}/pnomotoraks_unet.pth'

SAM2_REPO_DIR = '/content/sam2'
SAM2_CHECKPOINT = '/content/sam2/checkpoints/sam2.1_hiera_small.pt'
SAM2_CONFIG = 'configs/sam2.1/sam2.1_hiera_s.yaml'

IMAGE_SIZE = 224
PATCH_STRIDE = 224
SEED = 42
PNEUMOTHORAX_LABEL = 1
LUNG_LABEL = 2
TRAIN_RATIO = 0.65
VAL_RATIO = 0.10

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

unet_model = None
sam2_image_predictor = None
_sam2_base_model = None

# Segmentasyon modeli ve son isleme ayarlari. Varsayilan: 224x224 parca U-Net, esik 0.5, 80 pikselden kucuk
# bilesenleri at, yalnizca en buyuk bileseni tut. Dogrulamada secilen ayarlar load_segmenter_config() ile yuklenir.
SEG = {
    'kind': 'unet_patch',     # 'unet_patch' (parca U-Net) | 'guclu' (tam cozunurluklu ResNet34 U-Net)
    'threshold': 0.5,
    'min_area': 80,
    'keep': 'largest',        # 'largest' | 'all' | 'top2'
    'tta': False,             # yatay cevirme ile test-zamani artirma
    'lung_filter': False,     # yalnizca 'guclu': akciger olasiligi yuksek bolgeye dusen pnomotoraks bilesenlerini at
}
strong_model = None
LAST_LUNG_PROB = None


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


# ---------------------------------------------------------------------------
# Veri: hasta indeksi ve hasta bazli bolme 
# ---------------------------------------------------------------------------
def _slice_sort_key(path):
    stem = os.path.splitext(os.path.basename(path))[0]
    return (0, int(stem)) if stem.isdigit() else (1, stem)


def find_pairs_for_patient(patient_dir, numeric_order=False):
    # numeric_order=False: alfabetik siralama.
    # numeric_order=True : kesitleri sayisal sirada dondurur (video/sekans isleme icin;
    #                      '10.jpg' '2.jpg'den once gelmesin).
    pairs = []
    jpg_files = sorted(glob.glob(os.path.join(patient_dir, '*.jpg')))
    if numeric_order:
        jpg_files = sorted(jpg_files, key=_slice_sort_key)
    for jpg_path in jpg_files:
        stem = os.path.splitext(os.path.basename(jpg_path))[0]
        png_path = os.path.join(patient_dir, stem + '.png')
        if os.path.exists(png_path):
            pairs.append((jpg_path, png_path))
    return pairs


def build_patient_split(raw_dir=RAW_DIR, seed=SEED, numeric_order=True):
    """Sabit tohumla hasta bazli bolme uretir.

    Donus: patient_index {pid: [(jpg, png), ...]}, split_of_patient {pid: 'train'|'val'|'test'}
    """
    patient_dirs = sorted([d for d in glob.glob(os.path.join(raw_dir, '*')) if os.path.isdir(d)])
    patient_index = {}
    for pdir in patient_dirs:
        pid = os.path.basename(pdir)
        pairs = find_pairs_for_patient(pdir, numeric_order=numeric_order)
        if len(pairs) > 0:
            patient_index[pid] = pairs

    # Uyari: yalnizca kucuk harfli .jpg/.png ciftleri kullanilir. .jpeg veya .PNG gibi uzantilar Linux/Colab'da
    # eslesmez ve atlanir. Atlanan dosyalar burada listelenir.
    atlanan = [os.path.relpath(f, raw_dir) for d in patient_dirs for f in glob.glob(os.path.join(d, '*'))
               if not f.endswith(('.jpg', '.png'))]
    if atlanan:
        print(f'UYARI: standart disi uzantili {len(atlanan)} dosya kullanilmadi (ornek: {atlanan[:3]})')

    patient_ids = sorted(patient_index.keys())
    rng = random.Random(seed)
    rng.shuffle(patient_ids)
    n = len(patient_ids)
    n_train = int(round(n * TRAIN_RATIO))
    n_val = int(round(n * VAL_RATIO))
    split_of_patient = {}
    for pid in patient_ids[:n_train]:
        split_of_patient[pid] = 'train'
    for pid in patient_ids[n_train:n_train + n_val]:
        split_of_patient[pid] = 'val'
    for pid in patient_ids[n_train + n_val:]:
        split_of_patient[pid] = 'test'
    return patient_index, split_of_patient


def load_image_rgb(image_path):
    image = cv2.imread(image_path, cv2.IMREAD_COLOR)
    return cv2.cvtColor(image, cv2.COLOR_BGR2RGB)


def load_label_map(mask_path):
    m = np.array(Image.open(mask_path))
    if m.ndim == 3:
        m = m[..., 0]
    return m


def load_binary_mask(mask_path, label=PNEUMOTHORAX_LABEL):
    return (load_label_map(mask_path) == label).astype(np.uint8)


# ---------------------------------------------------------------------------
# Morfolojik yardimcilar ve rapor yardimcilari
# ---------------------------------------------------------------------------
def clean_mask(mask, min_area=80):
    mask = (mask > 0).astype(np.uint8)
    labeled, num = ndimage.label(mask)
    out = np.zeros_like(mask, dtype=np.uint8)
    for i in range(1, num + 1):
        region = (labeled == i)
        if region.sum() >= min_area:
            out[region] = 1
    return out


def largest_component(mask):
    labeled, num = ndimage.label(mask)
    if num == 0:
        return mask.astype(np.uint8)
    sizes = ndimage.sum(mask, labeled, range(1, num + 1))
    return (labeled == (int(np.argmax(sizes)) + 1)).astype(np.uint8)


def keep_components(mask, keep=None):
    keep = SEG['keep'] if keep is None else keep
    mask = (mask > 0).astype(np.uint8)
    if keep == 'largest':
        return largest_component(mask)
    if keep == 'top2':
        labeled, num = ndimage.label(mask)
        if num <= 2:
            return mask
        sizes = ndimage.sum(mask, labeled, range(1, num + 1))
        top = np.argsort(sizes)[::-1][:2] + 1
        return np.isin(labeled, top).astype(np.uint8)
    return mask


def postprocess_mask(mask, min_area=None, keep=None):
    """Ikili maskeye SEG ayarlarina gore kucuk bilesen temizligi + bilesen politikasi uygular."""
    return keep_components(clean_mask(mask, SEG['min_area'] if min_area is None else min_area), keep)


def area_ratio(mask):
    return float(mask.sum() / mask.size)


def estimate_side(mask, radiological_convention=True):
    """Pnomotoraksin tarafini tahmin eder.

    DIKKAT: Aksiyel BT goruntuleri radyolojik konvansiyonda gosterilir; goruntunun
    SOL yarisi hastanin SAG tarafidir. Varsayilan olarak radyolojik konvansiyon kullanilir.
    Goruntu cercevesine gore taraf gerekiyorsa radiological_convention=False verin.
    """
    if mask.sum() == 0:
        return 'belirgin degil'
    h, w = mask.shape
    img_left = mask[:, :w // 2].sum()
    img_right = mask[:, w // 2:].sum()
    if radiological_convention:
        side_of_img_left, side_of_img_right = 'sag', 'sol'
    else:
        side_of_img_left, side_of_img_right = 'sol', 'sag'
    if img_left > img_right * 1.2:
        return side_of_img_left
    elif img_right > img_left * 1.2:
        return side_of_img_right
    return 'bilateral'


def estimate_extent(mask):
    r = area_ratio(mask)
    if r < 0.005:
        return 'cok kucuk'
    elif r < 0.02:
        return 'kucuk'
    elif r < 0.06:
        return 'orta'
    return 'genis'


def make_overlay(image_np, mask, color=(255, 0, 0), alpha=0.35):
    overlay = image_np.copy()
    m = mask.astype(bool)
    col = np.array(color, dtype=np.float32)
    overlay[m] = ((1 - alpha) * overlay[m] + alpha * col).astype(np.uint8)
    return overlay


def mask_iou(mask_a, mask_b):
    inter = np.logical_and(mask_a > 0, mask_b > 0).sum()
    union = np.logical_or(mask_a > 0, mask_b > 0).sum()
    return 0.0 if union == 0 else float(inter) / float(union)


def mask_box(mask, margin=10):
    h, w = mask.shape
    ys, xs = np.where(mask > 0)
    return np.array([max(0, xs.min() - margin), max(0, ys.min() - margin),
                     min(w - 1, xs.max() + margin), min(h - 1, ys.max() + margin)])


# ---------------------------------------------------------------------------
# Metrikler
# ---------------------------------------------------------------------------
def segmentation_metrics(pred_mask, gt_mask, eps=1e-6):
    p = pred_mask.flatten().astype(np.uint8)
    g = gt_mask.flatten().astype(np.uint8)
    tp = int(np.sum((p == 1) & (g == 1)))
    tn = int(np.sum((p == 0) & (g == 0)))
    fp = int(np.sum((p == 1) & (g == 0)))
    fn = int(np.sum((p == 0) & (g == 1)))
    sens = (tp + eps) / (tp + fn + eps)
    prec = (tp + eps) / (tp + fp + eps)
    return {
        'dice': (2 * tp + eps) / (2 * tp + fp + fn + eps),
        'iou': (tp + eps) / (tp + fp + fn + eps),
        'accuracy': (tp + tn + eps) / (tp + tn + fp + fn + eps),
        'sensitivity': sens,
        'specificity': (tn + eps) / (tn + fp + eps),
        'precision': prec,
        'f1': (2 * prec * sens + eps) / (prec + sens + eps),
    }


def hd95_pixels(pred_mask, gt_mask):
    pred_mask = (pred_mask > 0).astype(np.uint8)
    gt_mask = (gt_mask > 0).astype(np.uint8)
    if pred_mask.sum() == 0 or gt_mask.sum() == 0:
        return float('nan')
    pb = pred_mask - ndimage.binary_erosion(pred_mask).astype(np.uint8)
    gb = gt_mask - ndimage.binary_erosion(gt_mask).astype(np.uint8)
    if pb.sum() == 0 or gb.sum() == 0:
        return float('nan')
    gt_dt = ndimage.distance_transform_edt(1 - gb)
    pr_dt = ndimage.distance_transform_edt(1 - pb)
    return float(max(np.percentile(gt_dt[pb > 0], 95), np.percentile(pr_dt[gb > 0], 95)))


def hemithorax_counts(pred_mask, gt_mask, min_pixels=50):
    """Yari-toraks (goruntunun sol/sag yarisi) duzeyinde tespit sayimlari.

    Veri setindeki kesitlerin neredeyse tamami pnomotoraks pozitiftir (851/852), bu yuzden kesit
    duzeyinde ozgulluk olculemez. Pnomotoraks cogunlukla tek tarafli oldugundan, referans maskede pnomotoraks
    olmayan karsi yari-toraks dogal bir negatif ornektir; burada tahmin edilen maske varsa yanlis pozitiftir.
    Donus: {'tp','fn','tn','fp'} (iki yari uzerinden toplam)."""
    w = gt_mask.shape[1]
    c = {'tp': 0, 'fn': 0, 'tn': 0, 'fp': 0}
    for sl in (slice(0, w // 2), slice(w // 2, w)):
        g = gt_mask[:, sl].sum() >= min_pixels
        p = pred_mask[:, sl].sum() >= min_pixels
        c['tp' if g and p else 'fn' if g else 'fp' if p else 'tn'] += 1
    return c


def dice_3d(pred_vol, gt_vol, eps=1e-6):
    p = pred_vol > 0
    g = gt_vol > 0
    inter = np.logical_and(p, g).sum()
    return float((2 * inter + eps) / (p.sum() + g.sum() + eps))


def bootstrap_ci(values, n_boot=2000, alpha=0.05, seed=SEED):
    values = np.asarray([v for v in values if not np.isnan(v)])
    if len(values) == 0:
        return float('nan'), float('nan'), float('nan')
    rng = np.random.default_rng(seed)
    means = [rng.choice(values, len(values), replace=True).mean() for _ in range(n_boot)]
    return float(values.mean()), float(np.percentile(means, 100 * alpha / 2)), float(np.percentile(means, 100 * (1 - alpha / 2)))


# ---------------------------------------------------------------------------
# U-Net
# ---------------------------------------------------------------------------
def init_unet(weight_path=UNET_WEIGHT_PATH):
    global unet_model
    import segmentation_models_pytorch as smp
    unet_model = smp.Unet(encoder_name='mobilenet_v2', encoder_weights=None,
                          in_channels=3, classes=1, activation=None).to(DEVICE)
    unet_model.load_state_dict(torch.load(weight_path, map_location=DEVICE))
    unet_model.eval()
    print('U-Net yuklendi:', weight_path)
    return unet_model


_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)


def _to_tensor(patch_rgb):
    x = (patch_rgb.astype(np.float32) / 255.0 - _MEAN) / _STD
    return torch.from_numpy(x.transpose(2, 0, 1)).unsqueeze(0)


def unet_prob_map(image_np, tta=None):
    """Aktif segmentasyon modeliyle tam goruntu pnomotoraks olasilik haritasi (HxW, 0-1).
    SEG['kind'] == 'guclu' ise tam cozunurluk modeli, aksi halde 224x224 parca U-Net kullanilir."""
    global LAST_LUNG_PROB
    tta = SEG['tta'] if tta is None else tta
    fn = _strong_prob if SEG['kind'] == 'guclu' else _patch_unet_prob_map
    p, lung = fn(image_np)
    if tta:
        p2, lung2 = fn(np.ascontiguousarray(image_np[:, ::-1]))
        p = (p + p2[:, ::-1]) / 2
        if lung is not None:
            lung = (lung + lung2[:, ::-1]) / 2
    LAST_LUNG_PROB = lung
    return p


@torch.no_grad()
def _strong_prob(image_np):
    h, w = image_np.shape[:2]
    H, W = int(np.ceil(h / 32) * 32), int(np.ceil(w / 32) * 32)
    img = cv2.copyMakeBorder(image_np, 0, H - h, 0, W - w, cv2.BORDER_CONSTANT, value=0)
    x = _to_tensor(img).to(DEVICE)
    with torch.autocast('cuda', enabled=(DEVICE == 'cuda')):
        out = torch.sigmoid(strong_model(x).float())[0].cpu().numpy()
    lung = out[1, :h, :w] if out.shape[0] > 1 else None
    return out[0, :h, :w], lung


def init_strong_model(weight_path, arch='Unet', encoder='resnet34', classes=2):
    global strong_model
    import segmentation_models_pytorch as smp
    strong_model = getattr(smp, arch)(encoder_name=encoder, encoder_weights=None, in_channels=3, classes=classes).to(DEVICE)
    strong_model.load_state_dict(torch.load(weight_path, map_location=DEVICE))
    strong_model.eval()
    SEG['kind'] = 'guclu'
    print('Guclu segmentasyon modeli yuklendi:', weight_path, arch, encoder)
    return strong_model


def load_segmenter_config(path):
    """Segmenter ayar dosyasini (model agirligi + dogrulamada secilen son isleme) yukler."""
    import json as _json
    cfg = _json.load(open(path))
    if cfg.get('kind') == 'guclu':
        init_strong_model(cfg['weight_path'], cfg.get('arch', 'Unet'), cfg.get('encoder', 'resnet34'), cfg.get('classes', 2))
    elif unet_model is None:
        init_unet()
    for k in ('kind', 'threshold', 'min_area', 'keep', 'tta', 'lung_filter'):
        if k in cfg:
            SEG[k] = cfg[k]
    print('Segmenter ayari:', SEG)
    return cfg


@torch.no_grad()
def _patch_unet_prob_map(image_np, patch_size=IMAGE_SIZE, stride=PATCH_STRIDE, batch_size=16):
    """Kaydirmali pencere ile tam goruntu olasilik haritasi (HxW, 0-1) (parca U-Net)."""
    h, w = image_np.shape[:2]
    ys = [0] if h <= patch_size else sorted(set(list(range(0, h - patch_size, stride)) + [h - patch_size]))
    xs = [0] if w <= patch_size else sorted(set(list(range(0, w - patch_size, stride)) + [w - patch_size]))
    prob_map = np.zeros((h, w), dtype=np.float32)
    count_map = np.zeros((h, w), dtype=np.float32)
    coords, tensors = [], []
    for y in ys:
        for x in xs:
            patch = image_np[y:y + patch_size, x:x + patch_size]
            ph, pw = patch.shape[:2]
            if ph < patch_size or pw < patch_size:
                patch = cv2.copyMakeBorder(patch, 0, patch_size - ph, 0, patch_size - pw, cv2.BORDER_CONSTANT, value=0)
            coords.append((y, x, ph, pw))
            tensors.append(_to_tensor(patch))
    for i in range(0, len(tensors), batch_size):
        batch = torch.cat(tensors[i:i + batch_size]).to(DEVICE)
        probs = torch.sigmoid(unet_model(batch))[:, 0].cpu().numpy()
        for (y, x, ph, pw), pr in zip(coords[i:i + batch_size], probs):
            prob_map[y:y + ph, x:x + pw] += pr[:ph, :pw]
            count_map[y:y + ph, x:x + pw] += 1.0
    count_map[count_map == 0] = 1.0
    return prob_map / count_map, None


def mask_from_prob(prob, lung_prob=None, threshold=None, min_area=None, keep=None, lung_filter=None):
    threshold = SEG['threshold'] if threshold is None else threshold
    lung_filter = SEG['lung_filter'] if lung_filter is None else lung_filter
    m = clean_mask((prob > threshold).astype(np.uint8), SEG['min_area'] if min_area is None else min_area)
    if lung_filter and lung_prob is not None and m.sum() > 0:
        # Pnomotoraks akciger parankiminin DISINDA (plevral bosluk) yer alir; akciger olasiligi
        # yuksek bolgeyle cogunlukla ortusen bilesenler yanlis pozitif kabul edilir.
        lab, n = ndimage.label(m)
        for j in range(1, n + 1):
            reg = lab == j
            if lung_prob[reg].mean() > 0.5:
                m[reg] = 0
    return keep_components(m, keep)


def unet_predict(image_np, threshold=None):
    """Cikti: (maske, guven, olasilik haritasi). Son isleme SEG ayarlarina gore yapilir."""
    prob = unet_prob_map(image_np)
    mask = mask_from_prob(prob, LAST_LUNG_PROB, threshold=threshold)
    return mask, float(prob.max()), prob


# ---------------------------------------------------------------------------
# SAM2 (goruntu modu)
# ---------------------------------------------------------------------------
def _sam2_import():
    import sys
    if SAM2_REPO_DIR not in sys.path:
        sys.path.insert(0, SAM2_REPO_DIR)


def init_sam2_image():
    global sam2_image_predictor, _sam2_base_model
    _sam2_import()
    from sam2.build_sam import build_sam2
    from sam2.sam2_image_predictor import SAM2ImagePredictor
    _sam2_base_model = build_sam2(config_file=SAM2_CONFIG, ckpt_path=SAM2_CHECKPOINT, device=DEVICE)
    sam2_image_predictor = SAM2ImagePredictor(_sam2_base_model)
    print('SAM2 (goruntu modu) hazir.')
    return sam2_image_predictor


def init_sam2_video(offload_video_to_cpu=False):
    _sam2_import()
    from sam2.build_sam import build_sam2_video_predictor
    vp = build_sam2_video_predictor(SAM2_CONFIG, SAM2_CHECKPOINT, device=DEVICE)
    print('SAM2 (video modu) hazir.')
    return vp


@torch.no_grad()
def sam2_set_image(image_np):
    sam2_image_predictor.set_image(image_np)


@torch.no_grad()
def sam2_predict(image_np, unet_mask, prompt_type='box', box_margin=10, multimask_select='score',
                 image_already_set=False):
    """SAM2 goruntu modu tahmini.

    prompt_type: 'box' | 'point' | 'box+point'
    multimask_select: 'score' (en yuksek SAM skoru) |
                      'agreement' (U-Net maskesiyle en yuksek IoU'lu aday)
    SEG['keep'] != 'largest' ise U-Net maskesinin her bagli bileseni (or. iki tarafli pnomotoraks)
    icin ayri istem verilir ve sonuclar birlestirilir.
    """
    if not image_already_set:
        sam2_image_predictor.set_image(image_np)
    if SEG['keep'] != 'largest' and unet_mask.sum() > 0:
        lab, n = ndimage.label(unet_mask > 0)
        if n > 1:
            out, skorlar, alanlar = np.zeros_like(unet_mask, dtype=np.uint8), [], []
            for j in range(1, n + 1):
                comp = (lab == j).astype(np.uint8)
                m, s = _sam2_tek(image_np, comp, prompt_type, box_margin, multimask_select)
                out = np.logical_or(out, m).astype(np.uint8)
                skorlar.append(s); alanlar.append(comp.sum())
            return keep_components(out), float(np.average(skorlar, weights=alanlar))
    return _sam2_tek(image_np, unet_mask, prompt_type, box_margin, multimask_select)


def _sam2_tek(image_np, unet_mask, prompt_type, box_margin, multimask_select):
    h, w = image_np.shape[:2]
    has_mask = unet_mask.sum() > 0
    box = mask_box(unet_mask, box_margin) if has_mask else np.array([int(w * .05), int(h * .05), int(w * .95), int(h * .95)])
    if has_mask:
        ys, xs = np.where(unet_mask > 0)
        # Agirlik merkezi maskenin disina dusebilir (hilal seklindeki pnomotoraks);
        # bu yuzden maske icinde merkeze en yakin piksel kullanilir.
        cy, cx = ys.mean(), xs.mean()
        j = int(np.argmin((ys - cy) ** 2 + (xs - cx) ** 2))
        point = np.array([[int(xs[j]), int(ys[j])]])
    else:
        point = np.array([[w // 2, h // 2]])

    kwargs = dict(multimask_output=True)
    if prompt_type == 'box':
        kwargs.update(point_coords=None, point_labels=None, box=box)
    elif prompt_type == 'point':
        kwargs.update(point_coords=point, point_labels=np.array([1]), box=None)
    else:
        kwargs.update(point_coords=point, point_labels=np.array([1]), box=box)
    masks, scores, _ = sam2_image_predictor.predict(**kwargs)

    # Tek bir bilesen/istem icin SAM2 cikti adayinin en buyuk bileseni alinir.
    cands = [largest_component(clean_mask(m.astype(np.uint8))) for m in masks]
    if multimask_select == 'agreement' and has_mask:
        best = int(np.argmax([mask_iou(c, unet_mask) for c in cands]))
    else:
        best = int(np.argmax(scores))
    return cands[best], float(scores[best])


def combine_masks(unet_mask, sam_mask, min_agreement_iou=0.3):
    # Kural: U-Net negatifse bos, uyum dusukse U-Net maskesine geri dus.
    # Bilesen politikasi SEG['keep'] ile belirlenir (varsayilan 'largest').
    if unet_mask.sum() == 0:
        return np.zeros_like(sam_mask, dtype=np.uint8)
    if mask_iou(unet_mask, sam_mask) < min_agreement_iou:
        return postprocess_mask(unet_mask)
    return postprocess_mask(np.logical_or(unet_mask > 0, sam_mask > 0).astype(np.uint8))


def compute_reliability(unet_conf, sam_conf, final_mask, wU=0.35, wS=0.45, wA=0.20):
    area_term = min(area_ratio(final_mask) / 0.03, 1.0)
    return float(max(0.0, min(1.0, wU * unet_conf + wS * sam_conf + wA * area_term)))


def baseline_pipeline(image_np):
    """Tek gecisli hat: U-Net -> SAM2 kutu istemi -> birlestirme."""
    t0 = time.time()
    unet_mask, unet_conf, prob = unet_predict(image_np)
    sam_mask, sam_conf = sam2_predict(image_np, unet_mask, prompt_type='box', box_margin=10)
    final = combine_masks(unet_mask, sam_mask)
    return {
        'mask': final, 'unet_mask': unet_mask, 'sam_mask': sam_mask,
        'unet_conf': unet_conf, 'sam_conf': sam_conf, 'prob': prob,
        'reliability': compute_reliability(unet_conf, sam_conf, final),
        'time_s': time.time() - t0,
    }


# ---------------------------------------------------------------------------
# Anatomik olabilirlik yardimcilari (dogrulama denetimleri icin)
# ---------------------------------------------------------------------------
def body_mask(image_np, thr=20):
    """Govde (vucut) maskesi: goruntu esiklenir, en buyuk bilesen alinir, delikleri doldurulur.
    Pnomotoraks havasi govde ICINDE, hasta disindaki hava govde DISINDA kalir."""
    gray = cv2.cvtColor(image_np, cv2.COLOR_RGB2GRAY) if image_np.ndim == 3 else image_np
    m = (gray > thr).astype(np.uint8)
    m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, np.ones((15, 15), np.uint8))
    m = largest_component(m)
    return ndimage.binary_fill_holes(m).astype(np.uint8)


def mean_intensity_in_mask(image_np, mask):
    gray = cv2.cvtColor(image_np, cv2.COLOR_RGB2GRAY) if image_np.ndim == 3 else image_np
    return float(gray[mask > 0].mean()) if mask.sum() > 0 else float('nan')


# ---------------------------------------------------------------------------
# VLM (Qwen2-VL)
# ---------------------------------------------------------------------------
vlm_model = None
vlm_processor = None
VLM_MODEL_ID = 'Qwen/Qwen2-VL-2B-Instruct'


def init_vlm(model_id=VLM_MODEL_ID):
    global vlm_model, vlm_processor, VLM_MODEL_ID
    from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
    VLM_MODEL_ID = model_id
    vlm_model = Qwen2VLForConditionalGeneration.from_pretrained(model_id, torch_dtype=torch.float16, device_map='auto')
    vlm_processor = AutoProcessor.from_pretrained(model_id)
    print('VLM hazir:', model_id)


@torch.no_grad()
def vlm_generate(image_pil, prompt_text, max_new_tokens=400, temperature=0.6, seed=None):
    from qwen_vl_utils import process_vision_info
    if seed is not None:
        torch.manual_seed(seed)
    messages = [{'role': 'user', 'content': [{'type': 'image', 'image': image_pil}, {'type': 'text', 'text': prompt_text}]}]
    chat = vlm_processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = vlm_processor(text=[chat], images=image_inputs, videos=video_inputs, padding=True, return_tensors='pt').to(DEVICE)
    out = vlm_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=True,
                             temperature=temperature, top_p=0.9, repetition_penalty=1.3)
    trimmed = [o[len(i):] for i, o in zip(inputs.input_ids, out)]
    return vlm_processor.batch_decode(trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0].strip()

In [ ]:
RUN_TOHUM = True   # False yapilirsa yalnizca hata skoru calisir

import os, sys, time, random
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from scipy import ndimage, stats
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

assert torch.cuda.is_available(), 'GPU yok: Calisma zamani turunu GPU yapip yeniden baslatin'
print('GPU:', torch.cuda.get_device_name(0))

from google.colab import drive
drive.mount('/content/drive')
sys.path.insert(0, '/content')
import pnx_ortak as po

po.set_seed()
D = '/content/drive/MyDrive'


def bul(*yollar):
    # Ilk calistirmadaki ve yayin surumundeki Drive klasor adlarinin ikisi de denenir
    for y in yollar:
        if os.path.exists(y):
            print('Bulundu:', y)
            return y
    raise FileNotFoundError('Hicbiri bulunamadi, yolu elle girin: ' + ' | '.join(yollar))


AGIRLIK_TEK = bul(f'{D}/pnx_calisma1b/guclu_Unet_resnet34_akcigersiz.pth', f'{D}/pnx_taban/guclu_Unet_resnet34_akcigersiz.pth')
AGIRLIK_COK = bul(f'{D}/pnx_calisma1b/guclu_Unet_resnet34.pth', f'{D}/pnx_taban/guclu_Unet_resnet34.pth')
SONUC_CSV = bul(f'{D}/pnx_calisma1_tekgorevli/test_tum_yontemler_kesit.csv',
                f'{D}/pnx_ajan_tek_gorevli_ikinci_gorus/test_tum_yontemler_kesit.csv')
OUT_DIR = f'{D}/pnx_ek_analizler'
os.makedirs(OUT_DIR, exist_ok=True)

patient_index, split_of_patient = po.build_patient_split()
SPLIT = {s: sorted(p for p, ss in split_of_patient.items() if ss == s) for s in ('train', 'val', 'test')}
KESIT = {s: [(pid, os.path.basename(j), j, m) for pid in SPLIT[s] for j, m in patient_index[pid]] for s in SPLIT}
HAVUZ = ThreadPoolExecutor(8)
IMG = {s: list(HAVUZ.map(po.load_image_rgb, [k[2] for k in KESIT[s]])) for s in ('val', 'test')}
GT = {s: list(HAVUZ.map(po.load_binary_mask, [k[3] for k in KESIT[s]])) for s in ('val', 'test')}
assert (len(SPLIT['test']), len(KESIT['test'])) == (43, 211), 'Test seti makaledeki bolmeyle uyusmuyor'
print({s: (len(SPLIT[s]), len(KESIT[s])) for s in SPLIT})


def kume_boot(df, f, n=2000, seed=42):
    # Hasta kumeli yuzdelik bootstrap. Hata kesiti icermeyen yeniden ornekler yeniden cekilir.
    r = np.random.default_rng(seed)
    hastalar = df.patient_id.unique()
    g = {p: df[df.patient_id == p] for p in hastalar}
    out = []
    while len(out) < n:
        o = pd.concat([g[p] for p in r.choice(hastalar, len(hastalar))])
        if 0 < o.hata.sum() < len(o):
            out.append(f(o))
    return np.percentile(out, [2.5, 97.5])

## 1. Dogrulamada tanimli hata skoru

Ana model ve ayar degismez (tek gorevli, TTA, esik 0,3, en kucuk alan 20, ilk iki bilesen). Iki oznitelik kullanilir: max(P) ve
katsayidaki alan terimi A (alan orani / 0,03, 1 ile sinirli). Lojistik regresyon yalnizca dogrulama kesitlerinde egitilir. Inceleme
esigi de dogrulamada, hatali kesitlerin en az %80'ini yakalayacak en dusuk deger olarak sabitlenir ve testte degistirilmez.

In [ ]:
po.init_strong_model(AGIRLIK_TEK, 'Unet', 'resnet34', classes=1)
CFG = dict(threshold=0.3, min_area=20, keep='top2')


def ozellik(s):
    rows = []
    for (pid, sl, _, _), im, g in zip(KESIT[s], IMG[s], GT[s]):
        p = po.unet_prob_map(im, tta=True).astype(np.float32)
        m = po.keep_components(po.mask_from_prob(p, None, threshold=CFG['threshold'], min_area=CFG['min_area'], keep='all', lung_filter=False), CFG['keep'])
        rows.append({'patient_id': pid, 'slice': sl, 'max_p': float(p.max()), 'A': min(float(m.mean()) / 0.03, 1.0),
                     'bos': bool(m.sum() == 0), 'dice': po.segmentation_metrics(m, g)['dice']})
    return pd.DataFrame(rows)


V, T = ozellik('val'), ozellik('test')
V['hata'], T['hata'] = (V.dice < 0.5).astype(int), (T.dice < 0.5).astype(int)
print(f"Dogrulama: {len(V)} kesit, {V.hata.sum()} hata, Dice {V.dice.mean():.3f} | Test: {len(T)} kesit, {T.hata.sum()} hata, Dice {T.dice.mean():.3f}")
if V.hata.sum() < 5:
    print('UYARI: dogrulamada hata sayisi cok az, skor belirsiz olabilir')

X = ['max_p', 'A']
lr = LogisticRegression(C=1.0, class_weight='balanced').fit(V[X], V.hata)
print('Lojistik katsayilar:', dict(zip(X, lr.coef_[0].round(3))), 'sabit', lr.intercept_.round(3))
# Skor buyudukce guven artar
V['yeni'], T['yeni'] = lr.decision_function(V[X]) * -1, lr.decision_function(T[X]) * -1

# Inceleme esigi yalnizca dogrulamada: hatalarin en az %80'ini yakalayan en dusuk esik
sv = np.sort(V.yeni[V.hata == 1].values)
ESIK = sv[int(np.ceil(0.8 * len(sv))) - 1] if len(sv) else np.nan
print('Dogrulamada sabitlenen inceleme esigi:', ESIK)

d = pd.read_csv(SONUC_CSV, dtype={'patient_id': str})
d = d[d.yontem == 'Yeniden deneme yok (yalniz isaretleme)'][['patient_id', 'slice', 'dice', 'guvenilirlik', 'insan_incelemesi']]
T = T.merge(d, on=['patient_id', 'slice'], suffixes=('', '_ana'))
assert len(T) == 211
T['R'] = T.guvenilirlik.fillna(0.0)
T['hata'] = (T.dice_ana < 0.5).astype(int)   # makaledeki hata tanimi (yalniz isaretleme)
print('Makaledeki yapilandirmada hata:', int(T.hata.sum()), '(beklenen 21)')


def auc(df, kol):
    return roc_auc_score(df.hata, -df[kol])


satir = []
for ad, alt in (('Tum kesitler', T), ('Bos olmayan tahmin', T[~T.bos])):
    for kol, etiket in (('yeni', 'Yeni skor (dogrulamada)'), ('max_p', 'max(P)'), ('R', 'Mevcut katsayi')):
        lo, hi = kume_boot(alt, lambda o: auc(o, kol))
        satir.append({'kume': ad, 'n': len(alt), 'hata': int(alt.hata.sum()), 'skor': etiket, 'AUROC': auc(alt, kol), 'GA alt': lo, 'GA ust': hi})
    for a, b in (('yeni', 'R'), ('yeni', 'max_p')):
        lo, hi = kume_boot(alt, lambda o: auc(o, a) - auc(o, b))
        satir.append({'kume': ad, 'n': len(alt), 'hata': int(alt.hata.sum()), 'skor': f'Fark {a} - {b}', 'AUROC': auc(alt, a) - auc(alt, b), 'GA alt': lo, 'GA ust': hi})
isaret = T.yeni <= ESIK
mevcut = T.insan_incelemesi.astype(str).str.lower().eq('true')
for ad, m in (('Yeni skor, dogrulama esigi', isaret), ('Mevcut bayrak', mevcut)):
    satir.append({'kume': 'Inceleme', 'n': 211, 'hata': int(T.hata.sum()), 'skor': ad, 'isaretli': int(m.sum()), 'isaretli orani': m.mean(),
                  'yakalanan hata': int((m & (T.hata == 1)).sum()), 'Dice isaretli': T.dice_ana[m].mean(), 'Dice isaretsiz': T.dice_ana[~m].mean()})


def sim(kol, oran):
    k = int(round(oran * len(T)))
    x = T.dice_ana.values.copy()
    x[np.argsort(T[kol].values)[:k]] = 1.0
    return x.mean()


for oran in (0.1, 0.2):
    for kol, etiket in (('yeni', 'Yeni skor'), ('max_p', 'max(P)'), ('R', 'Mevcut katsayi')):
        satir.append({'kume': f'Simulasyon %{int(oran * 100)}', 'skor': etiket, 'Dice sim': sim(kol, oran)})
S = pd.DataFrame(satir)
S.to_csv(f'{OUT_DIR}/TABLO_ek5_skor.csv', index=False)
T.drop(columns=['guvenilirlik']).to_csv(f'{OUT_DIR}/ek5_skor_test_kesit.csv', index=False)
display(S.round(4))

## 2. Uc tohum: tek gorevli ve cok gorevli model

Egitim ayarlari ana not defteriyle aynidir. Son isleme her tohumda dogrulamada ayni ozgun izgarayla secilir (esik 0,3-0,7, en kucuk
alan 20-150 piksel, bilesen politikasi, TTA var veya yok).

In [ ]:
if RUN_TOHUM:
    import segmentation_models_pytorch as smp
    import albumentations as A

    ARCH, ENCODER = 'Unet', 'resnet34'
    CROP, BATCH, EPOCHS, ITERS_PER_EPOCH, LR = 512, 8, 60, 150, 3e-4
    POS_CROP_P = 0.7
    IMG_TR = list(HAVUZ.map(po.load_image_rgb, [k[2] for k in KESIT['train']]))
    LAB_TR = list(HAVUZ.map(lambda k: po.load_label_map(k[3]).astype(np.uint8), KESIT['train']))
    aug = A.Compose([
        A.HorizontalFlip(p=0.5),
        A.Affine(scale=(0.9, 1.1), rotate=(-10, 10), translate_percent=(-0.05, 0.05), p=0.5),
        A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.4),
        A.RandomGamma(gamma_limit=(85, 115), p=0.3),
    ])


    class KirpmaVeri(Dataset):
        def __init__(self, n):
            self.n = n

        def __len__(self):
            return self.n

        def __getitem__(self, i):
            k = np.random.randint(len(IMG_TR))
            img, lab = IMG_TR[k], LAB_TR[k]
            h, w = lab.shape
            if np.random.rand() < POS_CROP_P and (lab == 1).any():
                ys, xs = np.where(lab == 1); j = np.random.randint(len(ys))
                cy, cx = ys[j] + np.random.randint(-CROP // 4, CROP // 4), xs[j] + np.random.randint(-CROP // 4, CROP // 4)
            else:
                cy, cx = np.random.randint(h), np.random.randint(w)
            y0 = int(np.clip(cy - CROP // 2, 0, max(h - CROP, 0))); x0 = int(np.clip(cx - CROP // 2, 0, max(w - CROP, 0)))
            im = img[y0:y0 + CROP, x0:x0 + CROP]; lb = lab[y0:y0 + CROP, x0:x0 + CROP]
            a = aug(image=im, mask=lb); im, lb = a['image'], a['mask']
            x = po._to_tensor(im)[0]
            y = torch.from_numpy(np.stack([lb == 1, lb == 2]).astype(np.float32))
            return x, y


    dice_l = smp.losses.DiceLoss('binary', from_logits=True)
    focal_l = smp.losses.FocalLoss('binary')


    def kayip(out, y):
        op, yp = out[:, 0:1].contiguous(), y[:, 0:1].contiguous()
        pnx = dice_l(op, yp) + focal_l(op, yp)
        if out.shape[1] == 1:
            return pnx
        oa, ya = out[:, 1:2].contiguous(), y[:, 1:2].contiguous()
        return pnx + 0.3 * (dice_l(oa, ya) + F.binary_cross_entropy_with_logits(oa, ya))


    @torch.no_grad()
    def val_dice_guclu(model):
        model.eval(); po.strong_model = model
        d = []
        for im, g in zip(IMG['val'], GT['val']):
            p, _ = po._strong_prob(im)
            d.append(po.segmentation_metrics(po.mask_from_prob(p, None, threshold=0.5, min_area=80, keep='all', lung_filter=False), g)['dice'])
        model.train()
        return float(np.mean(d))


    def egit(yol, classes, tohum):
        if os.path.exists(yol):
            print('Kayitli model bulundu, egitim atlaniyor:', yol)
            return
        po.set_seed(tohum)
        model = getattr(smp, ARCH)(encoder_name=ENCODER, encoder_weights='imagenet', in_channels=3, classes=classes).to(po.DEVICE)
        dl = DataLoader(KirpmaVeri(ITERS_PER_EPOCH * BATCH), batch_size=BATCH, shuffle=True, num_workers=os.cpu_count(), pin_memory=True,
                        drop_last=True, persistent_workers=True, worker_init_fn=lambda w: np.random.seed(tohum + w + torch.initial_seed() % 1000))
        opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
        sch = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=EPOCHS * ITERS_PER_EPOCH, pct_start=0.05)
        scaler = torch.amp.GradScaler('cuda')
        best, gecici = -1, yol + '.gecici'
        for ep in range(EPOCHS):
            t0, top = time.time(), 0.0
            for x, y in dl:
                x, y = x.to(po.DEVICE, non_blocking=True), y.to(po.DEVICE, non_blocking=True)
                opt.zero_grad(set_to_none=True)
                with torch.autocast('cuda'):
                    out = model(x)
                loss = kayip(out.float(), y)
                scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sch.step()
                top += loss.item()
            vd = val_dice_guclu(model) if (ep + 1) % 2 == 0 or ep == EPOCHS - 1 else np.nan
            if not np.isnan(vd) and vd > best:
                best = vd; torch.save(model.state_dict(), gecici)
            print(f'tohum {tohum} sinif {classes} epok {ep + 1}/{EPOCHS} kayip={top / ITERS_PER_EPOCH:.4f} val={vd:.4f} en_iyi={best:.4f} ({time.time() - t0:.0f}s)')
        os.replace(gecici, yol)   # yarim kalan egitim tamamlanmis sayilmasin
        del model, dl
        torch.cuda.empty_cache()

In [ ]:
ESIKLER, ALANLAR, POLITIKA = [0.3, 0.4, 0.5, 0.6, 0.7], [20, 50, 80, 150], ['largest', 'top2', 'all']


def hizli_izgara(probs, gts, eps=1e-6):
    # ek4 ile ayni hizli hesap: bilesen alani ve kesisim bir kez sayilir
    top = {}
    for thr in ESIKLER:
        for p, g in zip(probs, gts):
            lab, n = ndimage.label(p.astype(np.float32) > thr)
            alan = np.bincount(lab.ravel(), minlength=n + 1)[1:]
            kes = np.bincount(lab.ravel(), weights=g.ravel(), minlength=n + 1)[1:]
            gs = g.sum()
            for ma in ALANLAR:
                i = np.nonzero(alan >= ma)[0]
                a, k = alan[i], kes[i]
                for keep in POLITIKA:
                    if keep == 'largest':
                        j = [int(np.argmax(a))] if len(a) else []
                    elif keep == 'top2' and len(a) > 2:
                        j = np.argsort(a)[::-1][:2]
                    else:
                        j = slice(None)
                    top[(thr, ma, keep)] = top.get((thr, ma, keep), 0) + (2 * k[j].sum() + eps) / (a[j].sum() + gs + eps)
    return {c: v / len(probs) for c, v in top.items()}


def olasilik(im):
    p = po.unet_prob_map(im, tta=False)
    p2 = po.unet_prob_map(np.ascontiguousarray(im[:, ::-1]), tta=False)[:, ::-1]
    return p.astype(np.float16), ((p + p2) / 2).astype(np.float16)


def degerlendir(ad, tohum, yol, classes):
    po.init_strong_model(yol, ARCH_AD[0], ARCH_AD[1], classes=classes)
    po.SEG.update(kind='guclu')
    O = {False: {}, True: {}}
    for s in ('val', 'test'):
        c = [olasilik(im) for im in IMG[s]]
        O[False][s], O[True][s] = [x[0] for x in c], [x[1] for x in c]
    en = None
    for tta in (False, True):
        for (thr, ma, keep), d in hizli_izgara(O[tta]['val'], GT['val']).items():
            if en is None or d > en['val_dice']:
                en = dict(tta=tta, threshold=thr, min_area=ma, keep=keep, val_dice=d)
    rows = []
    for (pid, sl, _, _), p, g in zip(KESIT['test'], O[en['tta']]['test'], GT['test']):
        m = po.keep_components(po.mask_from_prob(p.astype(np.float32), None, threshold=en['threshold'], min_area=en['min_area'], keep='all', lung_filter=False), en['keep'])
        rows.append({'model': ad, 'tohum': tohum, 'patient_id': pid, 'slice': sl, 'dice': po.segmentation_metrics(m, g)['dice'],
                     'bos': bool(m.sum() == 0), **{'yt_' + a: b for a, b in po.hemithorax_counts(m, g).items()}})
    return pd.DataFrame(rows), en


if RUN_TOHUM:
    ARCH_AD = (ARCH, ENCODER)
    KOSULAR = [('Tek gorevli', 42, AGIRLIK_TEK, 1), ('Cok gorevli', 42, AGIRLIK_COK, 2)]
    for tohum in (7, 123):
        for ad, c in (('Tek gorevli', 1), ('Cok gorevli', 2)):
            yol = f"{OUT_DIR}/ek5_{'tek' if c == 1 else 'cok'}_tohum{tohum}.pth"
            egit(yol, c, tohum)
            KOSULAR.append((ad, tohum, yol, c))
    tum, ozet = [], []
    for ad, tohum, yol, c in KOSULAR:
        w, en = degerlendir(ad, tohum, yol, c)
        tum.append(w)
        hm = w.groupby('patient_id').dice.mean().mean()
        ozet.append({'model': ad, 'tohum': tohum, 'tta': en['tta'], 'esik': en['threshold'], 'en kucuk alan': en['min_area'], 'bilesen': en['keep'],
                     'val Dice': en['val_dice'], 'test Dice': w.dice.mean(), 'test Dice hasta ort.': hm, 'Bos tahmin': int(w.bos.sum()),
                     'YT ozgulluk': w.yt_tn.sum() / max(w.yt_tn.sum() + w.yt_fp.sum(), 1), 'YT yanlis pozitif': int(w.yt_fp.sum())})
        print(ozet[-1])
    W = pd.concat(tum)
    W.to_csv(f'{OUT_DIR}/ek5_tohum_test_kesit.csv', index=False)
    TO = pd.DataFrame(ozet)
    fark = []
    for tohum in sorted(W.tohum.unique()):
        a = W[(W.model == 'Tek gorevli') & (W.tohum == tohum)].set_index(['patient_id', 'slice']).dice
        b = W[(W.model == 'Cok gorevli') & (W.tohum == tohum)].set_index(['patient_id', 'slice']).dice.loc[a.index]
        pa, pb = a.groupby('patient_id').mean(), b.groupby('patient_id').mean()
        fark.append({'tohum': tohum, 'tek - cok': a.mean() - b.mean(), 'p kesit': stats.wilcoxon(a, b).pvalue, 'p hasta': stats.wilcoxon(pa, pb).pvalue})
    TF = pd.DataFrame(fark)
    TO.to_csv(f'{OUT_DIR}/TABLO_ek5_tohum.csv', index=False)
    OZ = TO.groupby('model', sort=False)[['val Dice', 'test Dice', 'YT ozgulluk', 'Bos tahmin']].agg(['mean', 'std'])
    OZ.to_csv(f'{OUT_DIR}/TABLO_ek5_tohum_ozet.csv')
    TF.to_csv(f'{OUT_DIR}/TABLO_ek5_tohum_fark.csv', index=False)
    display(TO.round(4)); display(OZ.round(4)); display(TF.round(4))
print('Bitti. Ciktilar:', OUT_DIR)

Tohum 42 satirlari makaledeki degerleri vermelidir (tek gorevli 0,769 ve 0,797, cok gorevli 0,748 ve 0,783).